# PTCG Merged Agent Workbench

Parent notebook for **The Pokémon Company - PTCG AI Battle Challenge Simulation**.

## Document map

| Doc | Notebook | Role |
|-----|----------|------|
| **4/8 — Dragapult** | `a-sample-rule-based-agent-dragapult-ex-deck.ipynb` | Base policy skeleton |
| **9/11 — Meta snapshot** | `pok-mon-tcg-ai-battle-meta-snapshot-07-july.ipynb` | Deck choice + holdout mindset |
| **10 — Expectimax** | `improved-probabilistic-agent.ipynb` | Search API + UCB1 + opponent reads |

## Path conventions

| | **Reads (input)** | **Writes (output)** |
|---|-------------------|---------------------|
| **Local** (`.venv`) | `data/` (e.g. `data/deck.csv`, `data/cg/`) | repo root + `notebooks/` |
| **Kaggle** | `/kaggle/input/` (attached datasets + competition files) | `/kaggle/working/` |

Section 1 resolves paths automatically via `env_paths.py`.

## Concrete merge plan

1. **Doc 4/8:** Dragapult scoring skeleton -> `DragapultPolicy`
2. **Docs 9/11:** meta-informed deck in `data/deck.csv` (local) or attached input dataset (Kaggle)
3. **Doc 10:** Search API + UCB1 on top of `DragapultPolicy` (not Lucario `AdvancedPolicy`)


## Research framing

### What type of agents?

You can use either pure rule-based or pure RL agents, but the interesting space here is **hybrid**:

- **Rule-based policy** as the backbone (handles complex card-game logic)
- **Search API** for lookahead
- Optionally a **learned value function** on top

Pure RL from scratch is very hard in this competition: games are long, rewards are sparse, and the action space is huge. Top public submissions are **rule-based + search** — closer to classical game AI (AlphaGo-style policy + search) than pure deep RL.

### Which domain to pick?

Target: **Search API integration + opponent-adaptive policy**.

Concrete stack for this repo:

1. Start from a strong public base agent (**Dragapult** in our builder; Lucario is the alternative in Expectimax).
2. Add real lookahead via the Search API (**UCB1** in `SEARCH_ALGO`).
3. Add **opponent archetype detection** (`_opponent_is_water_deck`, `_opponent_is_crustle_wall`) that shifts scoring weights mid-game.

This direction is underexplored in public notebooks, feasible on the competition timeline, and gives a clean research narrative.

### Publishable paper angle

Working title:

> *Opponent-Adaptive Search in Imperfect-Information Card Games*

Core question: how does **online opponent modeling + lookahead search** improve agents in a hidden-information combinatorial game?

This maps to game-AI literature (MCTS, PIMC — Perfect Information Monte Carlo), uses a real experimental setup (ladder + holdout results), and is novel enough for workshop venues such as **IEEE CoG (Conference on Games)** or **AAAI workshop tracks**.

### How this workbench maps to that story

| Layer | Source | Implemented as |
|-------|--------|----------------|
| Policy backbone | Doc 4/8 Dragapult | `DragapultPolicy` |
| Lookahead search | Doc 10 Expectimax | `SEARCH_ALGO` + UCB1 |
| Opponent adaptation | Doc 10 Expectimax | `_opponent_is_*` hooks in policy |
| Deck + validation mindset | Docs 9/11 Meta snapshot | Sections 2 and 5 in this notebook |


## Research & experimentation plan (2-week sprint)

Compressed from the original 8-week plan for `ptcg-adaptive-search-agent`. **Phase details are unchanged** — only the schedule is tighter.

### The core research question

> *Does opponent-adaptive heuristic search outperform static rule-based policy in an imperfect-information card game, and by how much does each component contribute?*

One thesis for both the paper and the Kaggle writeup.

### Two-week calendar (14 days)

| Days | Original window | Phase |
|------|-----------------|-------|
| 1–2 | Week 1–2 | Phase 1 — Baseline establishment |
| 3–4 | Week 2–3 | Phase 2 — Deck selection & meta analysis |
| 5–9 | Week 3–5 | Phase 3 — Ablation study |
| 10–11 | Week 4–5 | Phase 4 — Search depth analysis |
| 11–12 | Week 5–6 | Phase 5 — Opponent adaptation analysis |
| 13–14 | Week 6–7 | Phase 6 — Live ladder validation |

Days 11–12 overlap Phases 4 and 5 on purpose — run search sweeps in the morning, adaptation analysis in the afternoon.

---

### Phase 1 — Baseline establishment (Days 1–2)

**Goal:** Get a working submission and a reproducible evaluation harness.

**Tasks:**
- Submit the Dragapult-only policy (no search, no opponent detection) as **Baseline A**
- Submit the merged Dragapult + UCB1 Search agent as **Baseline B**
- Record ladder ratings for both
- Implement `run_holdout_suite()` properly — it needs to actually simulate games against a fixed panel of archetypes (Alakazam, Crustle, Spidops, Starmie) so you have offline results independent of the live ladder

**Why it matters for paper:** You need clean ablation starting points. Baseline A = no search. Baseline B = search but no opponent adaptation.

---

### Phase 2 — Deck selection & meta analysis (Days 3–4)

**Goal:** Pick and commit to a deck with a principled justification.

**Tasks:**
- Run your holdout suite with both Dragapult and Starmie decks against the four key archetypes
- Record win rates per matchup, not just overall
- Apply the meta snapshot logic: usage share vs. actual score rate — pick the deck that has positive edge against the current field composition, not just the highest raw win rate
- Document your deck selection decision with the actual numbers — this becomes Section 2 of your paper ("Deck Selection Under Meta Uncertainty")

**Decision point:** Commit to one deck before Phase 3. Changing decks mid-experimentation ruins your ablation story.

---

### Phase 3 — Ablation study (Days 5–9)

**Goal:** Isolate the contribution of each component. This is the heart of the paper.

Run four agent configurations against your fixed holdout panel:

| Agent Version | Search | Opponent Detection | Expected Role |
|---|---|---|---|
| V1 | ✗ | ✗ | Pure Dragapult baseline |
| V2 | ✓ UCB1 | ✗ | Search contribution |
| V3 | ✗ | ✓ Adaptive weights | Adaptation contribution |
| V4 | ✓ UCB1 | ✓ Adaptive weights | Full system |

Measure per matchup win rate, not just overall. You want to show that opponent detection helps specifically against Crustle/stall matchups, and search helps in tactical decision points.

**This is your Table 1 in the paper.**

---

### Phase 4 — Search depth analysis (Days 10–11)

**Goal:** Answer how much search depth actually matters given the time budget.

**Tasks:**
- Vary UCB1 candidate count: 4, 8, 12, 16 candidates
- Vary time budget: 0.5s, 1.0s, 1.5s, 2.0s per decision
- Plot win rate vs. compute budget curve
- Find the knee — where does additional search stop helping?

**Why it matters for paper:** This is your Figure 2. It directly addresses the practical question of compute-performance tradeoff in real-time game AI, which is a standard analysis in MCTS literature.

---

### Phase 5 — Opponent adaptation analysis (Days 11–12)

**Goal:** Show that adaptive weights matter and quantify how much.

**Tasks:**
- Against Crustle specifically: compare V1 vs V3 (Dragapult policy vs. Dragapult + Crustle-aware weights)
- Log which archetype was detected and when during games
- Track false positive rate — does the detector misidentify archetypes early game when the bench is empty?
- Consider adding one more archetype detector beyond what the Expectimax agent has (e.g., Spidops or Festival detection)

**Why it matters for paper:** Opponent modeling in hidden-information games is a known hard problem. Even a simple detector with real impact is a publishable contribution if properly measured.

---

### Phase 6 — Live ladder validation (Days 13–14)

**Goal:** Confirm offline results transfer to the live ladder.

**Tasks:**
- Submit V4 (full system) and record ladder rating over time
- Compare per-archetype matchup rates on the ladder vs. your holdout panel
- Measure the gap — if holdout says 70% vs Crustle but ladder shows 60%, that's a calibration finding worth reporting
- Apply the winner's curse check from the meta snapshot: if your holdout score drops significantly on a fresh panel, prefer the more stable configuration

---

### Paper structure (target: CoG 2027 or AAAI workshop)

| Section | Content | Source |
|---|---|---|
| Introduction | Imperfect-info card games as AI testbeds | Literature |
| Background | PTCG mechanics, PIMC, UCB1, opponent modeling | Literature |
| System Design | Dragapult policy + Search + Adaptation | Phase 1–2 |
| Deck Selection | Meta-informed deck choice methodology | Phase 2 |
| Experiments | Ablation table, search depth curve, adaptation analysis | Phase 3–5 |
| Live Evaluation | Ladder validation, holdout-to-live transfer gap | Phase 6 |
| Conclusion | What worked, what didn't, future work | All |

---

### One rule to follow throughout

**Document every decision as you make it, not at the end.** Every time you change a weight, add a detector, or switch a parameter — write one paragraph explaining why. That becomes your Kaggle writeup organically, and it becomes your paper's experimental section with almost no additional work.


## 1. Environment + path setup

- **Local:** run with the project `.venv` kernel; reads from `data/`.
- **Kaggle:** reads from `/kaggle/input/` (read-only); all generated files go to `/kaggle/working/`.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

for candidate in (Path.cwd() / "notebooks", Path.cwd()):
    if (candidate / "env_paths.py").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        break

from env_paths import describe_paths, discover_notebooks_dir, get_paths, stage_deck_for_build

NOTEBOOKS_DIR = discover_notebooks_dir()
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

PATHS = get_paths()
PATHS.ensure_dirs()

print(describe_paths(PATHS))
if PATHS.environment == "local" and not PATHS.using_local_venv:
    print("Tip: select the project .venv kernel for local runs.")

SOURCES = {
    "dragapult (Doc 4/8)": PATHS.ref_dir / "a-sample-rule-based-agent-dragapult-ex-deck.ipynb",
    "expectimax (Doc 10)": PATHS.ref_dir / "improved-probabilistic-agent.ipynb",
    "meta snapshot (Doc 9/11)": PATHS.ref_dir / "pok-mon-tcg-ai-battle-meta-snapshot-07-july.ipynb",
}

print("\nReference notebooks:")
for label, path in SOURCES.items():
    print(f"  {'OK' if path.exists() else 'MISSING'} - {label}: {path}")


## 2. Meta-informed deck choice (Docs 9/11)

Choose **Starmie** or **Festival Thwackey** for the meta-informed deck list. Agent code in the meta notebooks is base64-only (`main_b64`, `deck_b64`); this workbench uses their **analysis mindset** only.


In [ ]:
META_FIELD = pd.DataFrame([
    {"archetype": "starmie", "usage_pct": 13.85, "score_pct": 51.89, "role": "Primary meta candidate"},
    {"archetype": "festival_thwackey", "usage_pct": 1.16, "score_pct": 45.99, "role": "Low-share dark horse"},
    {"archetype": "dragapult", "usage_pct": 7.31, "score_pct": 49.13, "role": "Policy skeleton constants in builder"},
])

META_FIELD[META_FIELD["archetype"].isin(["starmie", "festival_thwackey"])].sort_values("usage_pct")


In [ ]:
if PATHS.deck_path and PATHS.deck_path.exists():
    deck = [int(line) for line in PATHS.deck_path.read_text().splitlines() if line.strip()]
    assert len(deck) == 60, f"Expected 60 cards, got {len(deck)}"
    print(f"Deck source ({PATHS.environment}): {PATHS.deck_path}")
    print(f"Cards: {len(deck)}, unique ids: {len(set(deck))}")
else:
    if PATHS.environment == "kaggle":
        print("Attach a dataset with deck.csv under /kaggle/input, then re-run.")
    else:
        print("Add data/deck.csv locally, then re-run.")


## 3. Source contributions

- **Doc 4/8:** `DragapultPolicy` — logs, deck reconstruction, combo planning, scoring
- **Doc 10:** `_opponent_is_water_deck`, `_opponent_is_crustle_wall`, `SEARCH_ALGO` + UCB1 (wired to Dragapult, not Lucario `AdvancedPolicy`)
- **Docs 9/11:** deck framing + holdout gates (sections 2 and 5)


## 4. Build merged `main.py`

Writes to `PATHS.main_py` (`/kaggle/working/main.py` on Kaggle, repo-root locally).


In [ ]:
import subprocess

builder = PATHS.notebooks_dir / "build_merged_agent.py"
if not builder.exists():
    builder = NOTEBOOKS_DIR / "build_merged_agent.py"

subprocess.run([sys.executable, str(builder)], check=True, cwd=str(builder.parent))

merged_path = PATHS.merged_main_py
if not merged_path.exists():
    merged_path = builder.parent / "merged_agent_main.py"

main_src = merged_path.read_text(encoding="utf-8")
PATHS.main_py.write_text(main_src, encoding="utf-8")
print(f"Wrote {PATHS.main_py} ({len(main_src.splitlines())} lines)")


In [ ]:
REQUIRED_MARKERS = {
    "DragapultPolicy": "Doc 4/8 policy skeleton",
    "_opponent_is_water_deck": "Doc 10 opponent read",
    "_opponent_is_crustle_wall": "Doc 10 opponent read",
    "SEARCH_ALGO": "Doc 10 search wrapper",
    "Phase 2: UCB1": "Doc 10 UCB1 loop",
    "search_begin": "Doc 10 Search API",
}

main_text = PATHS.main_py.read_text(encoding="utf-8")
checks = pd.DataFrame([
    {"marker": k, "purpose": v, "present": k in main_text}
    for k, v in REQUIRED_MARKERS.items()
])
missing = checks[~checks["present"]]["marker"].tolist()
print(checks.to_string(index=False))
if missing:
    raise RuntimeError(f"Build verification failed - missing: {missing}")
print("Build verification passed.")


## 5. Holdout validation (meta gates)


In [ ]:
HOLDOUT_OPPONENTS = ["starmie", "festival_thwackey", "hop_trevenant", "archaludon", "lucario"]
HOLDOUT_GAMES = 40
PROMOTE_THRESHOLD = 0.52


def run_holdout_suite(opponents=HOLDOUT_OPPONENTS, games=HOLDOUT_GAMES):
    raise NotImplementedError("Wire to cabt / kaggle-environments using PATHS.cg_dir")


def summarize_holdout(results):
    rows = []
    for row in results:
        total = row["wins"] + row["losses"] + row["ties"]
        rate = row["wins"] / total if total else 0.0
        passed = rate >= PROMOTE_THRESHOLD
        rows.append({
            **row,
            "win_rate": rate,
            "holdout_gate": "holdout_pass" if passed else "holdout_fail",
            "verdict": "PROMOTE_CANDIDATE" if passed else "HOLD_DO_NOT_SUBMIT",
        })
    return pd.DataFrame(rows)

print("Stress pool:", HOLDOUT_OPPONENTS)


## 6. Package submission

Output: `PATHS.submission_tar` (`/kaggle/working/submission.tar.gz` on Kaggle).

Reads `deck.csv` and `cg/` from input paths; writes tarball to working/output directory.


In [ ]:
import shutil
import tarfile


def build_submission(output: Path | None = None):
    output = output or PATHS.submission_tar
    deck_file = stage_deck_for_build(PATHS)

    if not PATHS.main_py.exists():
        raise FileNotFoundError(f"Run section 4 first. Missing: {PATHS.main_py}")

    cg_src = PATHS.cg_dir
    if cg_src is None or not (cg_src / "api.py").exists():
        raise FileNotFoundError(
            "cg SDK not found. Local: data/cg/. Kaggle: add competition sample_submission input."
        )

    staging = PATHS.output_root / ".submission_staging"
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)

    shutil.copy2(PATHS.main_py, staging / "main.py")
    shutil.copy2(deck_file, staging / "deck.csv")
    shutil.copytree(cg_src, staging / "cg")

    with tarfile.open(output, "w:gz") as tar:
        tar.add(staging / "main.py", arcname="main.py")
        tar.add(staging / "deck.csv", arcname="deck.csv")
        for item in sorted((staging / "cg").rglob("*")):
            if item.is_file() and "__pycache__" not in item.parts:
                tar.add(item, arcname=str(Path("cg") / item.relative_to(staging / "cg")))

    shutil.rmtree(staging)
    print(f"Created {output} ({output.stat().st_size / 1024 / 1024:.2f} MiB)")


# build_submission()


## 7. Checklist

1. Section 1 - verify environment + paths
2. Section 2 - deck.csv in `data/` (local) or `/kaggle/input/` (Kaggle)
3. Section 4 - build + verify `main.py`
4. Section 5 - holdout gate
5. Section 6 - `submission.tar.gz` from working/output directory
